In [1]:
from pathlib import Path

import pandas as pd
import mido

print("=" * 60)
print("         Required Libraries Imported Successfully")
print("=" * 60)

         Required Libraries Imported Successfully


## 1. Import the Required Libraries

This notebook accelerates previously generated MIDI files by modifying only the tempo information while preserving all musical events.

The following Python libraries are used throughout the notebook:

- **pathlib**: provides an object-oriented interface for managing project directories and file paths.
- **pandas**: summarizes the processing results in a structured table.
- **mido**: reads, edits, and writes Standard MIDI files, enabling tempo modification without altering the musical content.

These libraries establish the foundation for the MIDI tempo acceleration workflow.

## 2. Configure the Project Directories

This section automatically locates the root directory of the **TDC-Analysis-Book** project and defines the locations of the input and output MIDI datasets.

The notebook expects the following project structure:

```
TDC-Analysis-Book/
│
├── data/
│   └── generated_music/
│       ├── midi/
│       │   ├── data_availability_based_markov_model/
│       │   └── transition_probability_based_markov_model/
│       │
│       └── midi_accelerated/
│
└── webbook/
```

The notebook recursively searches all subdirectories of the input folder, allowing MIDI files generated by different Markov models to be processed automatically.

Accelerated MIDI files are written to a separate output directory, ensuring that the original generated files remain unchanged.

In [4]:
from pathlib import Path

# ============================================================
# Project directories
# ============================================================

PROJECT_ROOT = Path(r"D:\TDC-Analysis-Book")

INPUT_MIDI_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "generated_music"
    / "midi"
)

OUTPUT_MIDI_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "generated_music"
    / "midi_accelerated"
)

OUTPUT_MIDI_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 60)
print("           Project Directories Configured")
print("=" * 60)
print(f"Project root    : {PROJECT_ROOT}")
print(f"Input directory : {INPUT_MIDI_DIRECTORY}")
print(f"Output directory: {OUTPUT_MIDI_DIRECTORY}")

           Project Directories Configured
Project root    : D:\TDC-Analysis-Book
Input directory : D:\TDC-Analysis-Book\data\generated_music\midi
Output directory: D:\TDC-Analysis-Book\data\generated_music\midi_accelerated


In [5]:
# Locate all MIDI files recursively
midi_files = sorted(INPUT_MIDI_DIRECTORY.rglob("*.mid"))

print("=" * 60)
print("                MIDI Files Located")
print("=" * 60)
print(f"Total MIDI files : {len(midi_files):,}")

if len(midi_files) == 0:
    raise FileNotFoundError(
        f"No MIDI files were found in:\n{INPUT_MIDI_DIRECTORY}"
    )

print("\nFirst 10 MIDI files:\n")

for midi_file in midi_files[:10]:
    print(midi_file.relative_to(INPUT_MIDI_DIRECTORY))

if len(midi_files) > 10:
    print("\n...")

                MIDI Files Located
Total MIDI files : 8

First 10 MIDI files:

data_availability\AI_generated_data_availability_hicaz_duyek.mid
data_availability\AI_generated_data_availability_nihavent_duyek.mid
data_availability_based_markov_model\AI_generated_data_availability_based_markov_model_hicaz_duyek.mid
data_availability_based_markov_model\AI_generated_data_availability_based_markov_model_nihavent_duyek.mid
musical_diversity\AI_generated_musical_diversity_rast_duyek.mid
musical_diversity_based_markov_model\AI_generated_musical_diversity_based_markov_model_hicaz_duyek.mid
musical_diversity_based_markov_model\AI_generated_musical_diversity_based_markov_model_rast_duyek.mid
musical_diversity_based_markov_model\AI_generated_musical_diversity_based_markov_model_ussak_duyek.mid


## 4. Configure the Tempo Acceleration

This notebook modifies the playback speed of previously generated MIDI files.

The **SPEED_FACTOR** parameter controls how much faster each MIDI file should be played.

Only existing tempo metadata is modified. No new tempo values are introduced, ensuring that the generated MIDI files preserve the musical characteristics defined during the Markov chain generation process.

In [12]:
# ============================================================
# Tempo acceleration configuration
# ============================================================

# Playback speed multiplier
# 2.0 = twice as fast
# 1.5 = 50% faster
# 1.25 = 25% faster

SPEED_FACTOR = 2.0

print("=" * 60)
print("         Tempo Acceleration Configuration")
print("=" * 60)
print(f"Speed factor : {SPEED_FACTOR}x")

         Tempo Acceleration Configuration
Speed factor : 2.0x


## 5. Accelerate Existing Tempo Information

This function modifies only the existing tempo metadata contained in each MIDI file.

For every tempo event:

- the original tempo is read,
- the playback speed is increased according to the selected acceleration factor,
- the updated MIDI file is written to the output directory.

If a MIDI file does not contain tempo metadata, the notebook reports this information and saves the file unchanged.

No musical events—including pitches, note durations, velocities, instruments, or event ordering—are modified during this process.

In [13]:
def accelerate_midi(
    input_file: Path,
    output_file: Path,
    speed_factor: float
):
    """
    Create an accelerated version of an existing MIDI file by
    modifying only its tempo metadata.
    """

    midi = mido.MidiFile(input_file)

    tempo_found = False
    original_bpm = None
    accelerated_bpm = None

    for track in midi.tracks:

        for msg in track:

            if msg.type == "set_tempo":

                tempo_found = True

                original_bpm = round(
                    mido.tempo2bpm(msg.tempo),
                    2
                )

                msg.tempo = int(msg.tempo / speed_factor)

                accelerated_bpm = round(
                    mido.tempo2bpm(msg.tempo),
                    2
                )

    if not tempo_found:

        print(f"No tempo information found: {input_file.name}")

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    midi.save(output_file)

    return {
        "Original BPM": original_bpm,
        "Accelerated BPM": accelerated_bpm,
        "Tempo Found": tempo_found
    }

## 6. Accelerate All Generated MIDI Files

This section applies the tempo acceleration function to every MIDI file detected in the input directory.

For each source file, the notebook:

1. identifies the original model folder,
2. preserves the relative directory structure,
3. creates a new filename containing the selected speed factor,
4. generates an accelerated MIDI copy,
5. records the processing status.

The original MIDI files remain unchanged. Accelerated files are stored in the separate `midi_accelerated` directory.

In [14]:
processing_results = []

speed_label = str(SPEED_FACTOR).replace(".", "_")

for input_file in midi_files:

    relative_path = input_file.relative_to(INPUT_MIDI_DIRECTORY)

    output_filename = (
        f"{input_file.stem}_fast_{speed_label}x.mid"
    )

    output_file = (
        OUTPUT_MIDI_DIRECTORY
        / relative_path.parent
        / output_filename
    )

    try:

        result = accelerate_midi(
            input_file=input_file,
            output_file=output_file,
            speed_factor=SPEED_FACTOR
        )

        processing_results.append({
            "Model": str(relative_path.parent),
            "Input File": input_file.name,
            "Output File": output_filename,
            "Original BPM": result["Original BPM"],
            "Accelerated BPM": result["Accelerated BPM"],
            "Tempo Found": result["Tempo Found"],
            "Status": "Success"
        })

        print(f"✓ {relative_path}")

    except Exception as error:

        processing_results.append({
            "Model": str(relative_path.parent),
            "Input File": input_file.name,
            "Output File": output_filename,
            "Original BPM": None,
            "Accelerated BPM": None,
            "Tempo Found": False,
            "Status": str(error)
        })

        print(f"✗ {relative_path}")

✓ data_availability\AI_generated_data_availability_hicaz_duyek.mid
✓ data_availability\AI_generated_data_availability_nihavent_duyek.mid
✓ data_availability_based_markov_model\AI_generated_data_availability_based_markov_model_hicaz_duyek.mid
✓ data_availability_based_markov_model\AI_generated_data_availability_based_markov_model_nihavent_duyek.mid
✓ musical_diversity\AI_generated_musical_diversity_rast_duyek.mid
✓ musical_diversity_based_markov_model\AI_generated_musical_diversity_based_markov_model_hicaz_duyek.mid
✓ musical_diversity_based_markov_model\AI_generated_musical_diversity_based_markov_model_rast_duyek.mid
✓ musical_diversity_based_markov_model\AI_generated_musical_diversity_based_markov_model_ussak_duyek.mid


## 7. Review the Processing Summary

The processing results are converted into a Pandas DataFrame.

The summary table reports:

- the Markov model directory,
- the original MIDI filename,
- the accelerated MIDI filename,
- the applied speed factor,
- the processing status.

This table provides a transparent record of the batch conversion process and makes it easier to identify unsuccessful files.

In [15]:
summary_df = pd.DataFrame(processing_results)

print("=" * 70)
print("           MIDI Tempo Acceleration Summary")
print("=" * 70)

print(f"Processed MIDI files : {len(summary_df)}")

print(
    f"Tempo metadata found : "
    f"{summary_df['Tempo Found'].sum()}"
)

print(
    f"Files without tempo  : "
    f"{len(summary_df)-summary_df['Tempo Found'].sum()}"
)

summary_df

           MIDI Tempo Acceleration Summary
Processed MIDI files : 8
Tempo metadata found : 8
Files without tempo  : 0


,Model,Input File,Output File,Original BPM,Accelerated BPM,Tempo Found,Status
0,data_availability,AI_generated_data_availability_hicaz_duyek.mid,AI_generated_data_availability_hicaz_duyek_fas...,120.0,240.0,True,Success
1,data_availability,AI_generated_data_availability_nihavent_duyek.mid,AI_generated_data_availability_nihavent_duyek_...,120.0,240.0,True,Success
2,data_availability_based_markov_model,AI_generated_data_availability_based_markov_mo...,AI_generated_data_availability_based_markov_mo...,120.0,240.0,True,Success
3,data_availability_based_markov_model,AI_generated_data_availability_based_markov_mo...,AI_generated_data_availability_based_markov_mo...,120.0,240.0,True,Success
4,musical_diversity,AI_generated_musical_diversity_rast_duyek.mid,AI_generated_musical_diversity_rast_duyek_fast...,120.0,240.0,True,Success
5,musical_diversity_based_markov_model,AI_generated_musical_diversity_based_markov_mo...,AI_generated_musical_diversity_based_markov_mo...,120.0,240.0,True,Success
6,musical_diversity_based_markov_model,AI_generated_musical_diversity_based_markov_mo...,AI_generated_musical_diversity_based_markov_mo...,120.0,240.0,True,Success
7,musical_diversity_based_markov_model,AI_generated_musical_diversity_based_markov_mo...,AI_generated_musical_diversity_based_markov_mo...,120.0,240.0,True,Success


## 8. Save the Processing Report

The processing summary is saved as a CSV file in the accelerated MIDI directory.

This report documents which files were processed, which speed factor was applied, and whether each operation was completed successfully. It supports reproducibility and provides a permanent record of the tempo acceleration workflow.

In [16]:
REPORT_FILE = (
    OUTPUT_MIDI_DIRECTORY
    / "midi_tempo_acceleration_report.csv"
)

summary_df.to_csv(
    REPORT_FILE,
    index=False
)

print("=" * 60)
print("              Processing Report Saved")
print("=" * 60)
print(f"Report filename : {REPORT_FILE.name}")
print(f"Output directory: {OUTPUT_MIDI_DIRECTORY}")

              Processing Report Saved
Report filename : midi_tempo_acceleration_report.csv
Output directory: D:\TDC-Analysis-Book\data\generated_music\midi_accelerated


In [11]:
from pathlib import Path

for file in OUTPUT_MIDI_DIRECTORY.rglob("*.mid"):
    print(file)

D:\TDC-Analysis-Book\data\generated_music\midi_accelerated\data_availability\AI_generated_data_availability_hicaz_duyek_fast_2_0x.mid
D:\TDC-Analysis-Book\data\generated_music\midi_accelerated\data_availability\AI_generated_data_availability_nihavent_duyek_fast_2_0x.mid
D:\TDC-Analysis-Book\data\generated_music\midi_accelerated\data_availability_based_markov_model\AI_generated_data_availability_based_markov_model_hicaz_duyek_fast_2_0x.mid
D:\TDC-Analysis-Book\data\generated_music\midi_accelerated\data_availability_based_markov_model\AI_generated_data_availability_based_markov_model_nihavent_duyek_fast_2_0x.mid
D:\TDC-Analysis-Book\data\generated_music\midi_accelerated\musical_diversity\AI_generated_musical_diversity_rast_duyek_fast_2_0x.mid
D:\TDC-Analysis-Book\data\generated_music\midi_accelerated\musical_diversity_based_markov_model\AI_generated_musical_diversity_based_markov_model_hicaz_duyek_fast_2_0x.mid
D:\TDC-Analysis-Book\data\generated_music\midi_accelerated\musical_diversity_